# 2교시. OCR 기반 텍스트 추출 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/02_ocr_basic.ipynb)

## 오늘 꼭 할 일

공개 한국 영수증에 실제 OCR을 실행하고 원본 위 좌표·신뢰도를 확인합니다.

1. 제공 예제로 결과를 먼저 만듭니다.
2. 화면에서 이번 교시의 핵심 결과 한 가지를 확인합니다.
3. 시간이 남으면 다른 공개·비식별 자료로 반복하고 차이를 기록합니다.

**끝났다는 증거:** 화면의 `✅ 실습 완료`와
`course_outputs/ocr_result.json` 파일

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

필수 실습에서는 제공 예제를 사용합니다. 다른 자료를 사용한 경우에는
화면에 표시된 파일명이 내가 선택한 파일과 같은지 먼저 확인합니다.
모델 설치가 3분 이상 진행되지 않으면 실행을 중지하고 제공 예제로
핵심 단계를 계속합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 제공된 한국 영수증을 PaddleOCR로 직접 읽고 상자가 글자 위에 맞는지 확인합니다.
- **내가 바꾸는 곳:** `실습_자료`에서 제공 예제·파일 업로드·인터넷 이미지 URL 중 하나를 고릅니다.
- **인터넷 자료로 다시 실험:** 문서 한 장만 바꾸어 사진 기울기·해상도·언어에 따라 박스와 글자가 어떻게 달라지는지 기록합니다.

먼저 제공 예제로 끝까지 실행해 `✅ 실습 완료`를 확인하세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 만든 결과 파일은
3~7교시에 이어 쓸 수 있습니다. 매 교시 마지막의 **다른 자료 실험
기록**에서 잘된 점과 실패한 점을 남깁니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.
5. 길고 어려운 준비 코드는 접혀 있습니다. 제목 왼쪽의 화살표를 눌러
   펼칠 수 있지만, 처음에는 펼치지 않아도 됩니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
#@title 🟢 0. 실습 환경 준비 — 그대로 실행 { display-mode: "form" }
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 실습 완료 문구와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# `OUTPUT_DIR`는 모든 산출물의 공통 폴더입니다. 업로드·다운로드 함수와 자료 로더를 등록하는 준비 셀이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 7, '공통 환경 준비', '결과 폴더와 실습 자료 다운로드 기능을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '`OUTPUT_DIR`는 모든 산출물의 공통 폴더입니다. 업로드·다운로드 함수와 자료 로더를 등록하는 준비 셀이므로 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
AUTOMATED_CHECK = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or AUTOMATED_CHECK:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 자료 선택에서 "
            "'제공 예제'를 고르거나 파일을 다시 선택하세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if AUTOMATED_CHECK:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if AUTOMATED_CHECK:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 7, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
#@title 🔵 실습 자료 고르기 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `실습_자료`에서 제공 예제·내 파일·인터넷 이미지 주소 중 하나를 고릅니다. 처음에는 제공 예제로 실행하고, 다음 실행에서 자료만 바꿉니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 7, 'OCR 입력 한 장 준비', '공개 영수증과 장애 시 사용할 검수 데이터를 불러옵니다.', '직접 읽을 파일명과 처리 계획이 표시되어야 합니다.', '`실습_자료`에서 제공 예제·내 파일·인터넷 이미지 주소 중 하나를 고릅니다. 처음에는 제공 예제로 실행하고, 다음 실행에서 자료만 바꿉니다.', 'optional')

# INPUT_FORM_CELL
import hashlib
import io
import pandas as pd
import requests
from PIL import Image, ImageDraw
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

SAMPLE_IMAGE_PATH = (
    "sample_docs/public_receipts/korea/"
    "taebaek_restaurant_2025_redacted.png"
)
RECORDED_OCR_PATH = (
    "tests/fixtures/ppocrv5_recorded_receipt_tokens.json"
)
OCR_METADATA_PATH = (
    "tests/fixtures/ppocrv5_recorded_receipt_metadata.json"
)
lesson_assets = load_course_assets(
    SAMPLE_IMAGE_PATH,
    RECORDED_OCR_PATH,
    OCR_METADATA_PATH,
)
provided_image = Image.open(
    io.BytesIO(lesson_assets[SAMPLE_IMAGE_PATH])
).convert("RGB")
receipt_image = provided_image.copy()
PROVIDED_OCR_RESULT = json.loads(
    lesson_assets[RECORDED_OCR_PATH].decode("utf-8")
)
OCR_RECORD_METADATA = json.loads(
    lesson_assets[OCR_METADATA_PATH].decode("utf-8")
)
assert hashlib.sha256(
    lesson_assets[SAMPLE_IMAGE_PATH]
).hexdigest() == OCR_RECORD_METADATA["source_image_sha256"]
assert hashlib.sha256(
    lesson_assets[RECORDED_OCR_PATH]
).hexdigest() == OCR_RECORD_METADATA["token_file_sha256"]
assert provided_image.size == (
    OCR_RECORD_METADATA["source_image_size"]["width"],
    OCR_RECORD_METADATA["source_image_size"]["height"],
)
for item in PROVIDED_OCR_RESULT:
    item["confidence_source"] = "제공 예제의 이전 실제 OCR"

# TODO(선택): 제공 예제를 끝낸 뒤 자료 선택만 바꾸어 다시 실행하세요.
실습_자료 = "제공 예제" #@param ["제공 예제", "내 컴퓨터에서 업로드", "인터넷 이미지 URL"]
인터넷_이미지_URL = "" #@param {type:"string"}
if AUTOMATED_CHECK:
    실습_자료 = "제공 예제"

INPUT_FILE_NAME = "taebaek_restaurant_2025_redacted.png"
if 실습_자료 == "내 컴퓨터에서 업로드":
    from google.colab import files
    print(
        "JPG·JPEG·PNG·WEBP 사진 한 장을 선택하세요. "
        "인터넷 자료는 이용조건을 확인하고, 내 문서는 카드·전화·"
        "회원번호 등 식별정보를 먼저 가립니다."
    )
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("문서 이미지 한 장만 선택하세요.")
    INPUT_FILE_NAME, uploaded_bytes = next(iter(uploaded.items()))
    if Path(INPUT_FILE_NAME).suffix.lower() not in {
        ".jpg", ".jpeg", ".png", ".webp"
    }:
        raise ValueError(
            "2교시는 JPG·JPEG·PNG·WEBP 한 장을 사용합니다. "
            "PDF·Office 문서는 8교시에서 체험합니다."
        )
    receipt_image = Image.open(
        io.BytesIO(uploaded_bytes)
    ).convert("RGB")
elif 실습_자료 == "인터넷 이미지 URL":
    if not 인터넷_이미지_URL.strip():
        raise ValueError("인터넷_이미지_URL에 이미지 주소를 붙여 넣으세요.")
    response = requests.get(인터넷_이미지_URL.strip(), timeout=30)
    response.raise_for_status()
    if len(response.content) > 5 * 1024 * 1024:
        raise ValueError("이미지는 5MB 이하 한 장만 사용하세요.")
    try:
        receipt_image = Image.open(
            io.BytesIO(response.content)
        ).convert("RGB")
    except Exception as exc:
        raise ValueError(
            "웹페이지 주소가 아니라 JPG·PNG·WEBP 이미지 자체의 "
            "주소를 입력하세요."
        ) from exc
    from urllib.parse import urlparse
    INPUT_FILE_NAME = (
        Path(urlparse(인터넷_이미지_URL).path).name
        or "internet_document.png"
    )

READ_CURRENT_IMAGE = not AUTOMATED_CHECK
print(
    "선택한 자료:",
    실습_자료,
)
print(
    "이번 실행:",
    (
        "지금 이 사진을 OCR로 직접 읽습니다."
        if READ_CURRENT_IMAGE
        else "제공 예제로 실행 순서를 확인합니다."
    ),
)
print("입력 파일:", INPUT_FILE_NAME)

complete_lab_step(2, 7, '직접 읽을 파일명과 처리 계획이 표시되어야 합니다.')


## 실행

**PaddleOCR**는 실행 도구이고, **PP-OCRv5 Korean**은 그 안에서
사용하는 한국어 OCR 모델입니다. 이 노트북은 PaddleOCR 3.7에서
현재 이미지 한 장을 실제로 읽습니다.

설치·모델 다운로드가 3분을 넘기면 중지합니다. 오류 메시지를 보존한
채 같은 공개 영수증을 이전에 실제로 읽어 보존한 결과로 전환합니다.
이때 현재 사진을 분석한 결과가 아니라는 안내가 표시됩니다.


In [ ]:
#@title 🟢 PP-OCRv5 실행 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `READ_CURRENT_IMAGE`는 Colab에서 현재 이미지를 OCR로 읽게 합니다. 학생은 바꾸지 않고 `결과 출처`와 판독 영역 수만
# 확인합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 7, 'PP-OCRv5 실행', '영수증 한 장을 지금 OCR로 읽고, 실패하면 예제 결과임을 분명히 표시합니다.', '결과 출처·실패 이유·판독 영역 수를 확인합니다.', '`READ_CURRENT_IMAGE`는 Colab에서 현재 이미지를 OCR로 읽게 합니다. 학생은 바꾸지 않고 `결과 출처`와 판독 영역 수만 확인합니다.', 'none')

OCR_RESULT = PROVIDED_OCR_RESULT
RESULT_SOURCE = "제공 예제 사용"
OCR_ERROR = "자동검사용 실행"

if READ_CURRENT_IMAGE:
    import subprocess
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q",
             "paddlepaddle==3.2.1", "paddleocr==3.7.0"]
        )
        from paddleocr import PaddleOCR

        image_path = OUTPUT_DIR / "ocr_input.jpg"
        receipt_image.save(image_path)
        engine = PaddleOCR(
            lang="korean",
            ocr_version="PP-OCRv5",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            device="cpu",
        )
        page = list(engine.predict(str(image_path)))[0]
        payload = page.json() if callable(page.json) else page.json
        result = payload.get("res", payload)
        OCR_RESULT = [
            {
                "box": box.tolist() if hasattr(box, "tolist") else box,
                "text": text,
                "confidence": float(score),
            }
            for box, text, score in zip(
                result.get("rec_polys", []),
                result.get("rec_texts", []),
                result.get("rec_scores", []),
            )
        ]
        RESULT_SOURCE = "현재 이미지 직접 처리"
        OCR_ERROR = ""
    except Exception as exc:
        RESULT_SOURCE = "제공 예제 사용"
        OCR_ERROR = f"{type(exc).__name__}: {exc}"

if RESULT_SOURCE == "현재 이미지 직접 처리":
    print("결과 출처: 지금 선택한 사진을 직접 읽었습니다.")
else:
    print("결과 출처: 제공 예제의 OCR 결과를 사용합니다.")
    print("중요: 현재 사진을 분석한 결과가 아닙니다.")
if OCR_ERROR:
    print("직접 읽지 못한 이유:", OCR_ERROR)
if RESULT_SOURCE == "제공 예제 사용":
    if INPUT_FILE_NAME != "taebaek_restaurant_2025_redacted.png":
        print(
            "내 영수증을 직접 읽지 못해 박스 표시는 "
            "공개 영수증의 이전 실제 OCR 기록으로 바꿉니다."
        )
    receipt_image = provided_image.copy()
    DISPLAY_INPUT_FILE_NAME = "taebaek_restaurant_2025_redacted.png"
    OCR_COORDINATE_SIZE = (
        OCR_RECORD_METADATA["coordinate_space"]["width"],
        OCR_RECORD_METADATA["coordinate_space"]["height"],
    )
    expected_height = round(
        receipt_image.height
        * OCR_COORDINATE_SIZE[0]
        / receipt_image.width
    )
    assert OCR_COORDINATE_SIZE[1] == expected_height
else:
    DISPLAY_INPUT_FILE_NAME = INPUT_FILE_NAME
    OCR_COORDINATE_SIZE = receipt_image.size
print("판독 영역:", len(OCR_RESULT))

complete_lab_step(3, 7, '결과 출처·실패 이유·판독 영역 수를 확인합니다.')


In [ ]:
#@title 🟢 OCR 위치 시각화와 저장 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `OCR_RESULT`를 원본 위에 그리고 인식 글자·신뢰도 표를 바로 표시합니다. `RESULT_SOURCE`도 저장해 현재 이미지 결과인지 제공
# 예제인지 구분합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 7, 'OCR 위치 시각화와 저장', '판독 영역을 원본 위에 그리고 JSON과 이미지를 저장합니다.', '`✅ 실습 완료`와 두 결과 파일을 확인합니다.', '`OCR_RESULT`를 원본 위에 그리고 인식 글자·신뢰도 표를 바로 표시합니다. `RESULT_SOURCE`도 저장해 현재 이미지 결과인지 제공 예제인지 구분합니다.', 'none')

annotated = receipt_image.copy()
draw = ImageDraw.Draw(annotated)
scale_x = annotated.width / OCR_COORDINATE_SIZE[0]
scale_y = annotated.height / OCR_COORDINATE_SIZE[1]
for item in OCR_RESULT:
    scaled_points = [
        (
            round(point[0] * scale_x),
            round(point[1] * scale_y),
        )
        for point in item["box"]
    ]
    draw.line(
        scaled_points + [scaled_points[0]],
        fill="#0F766E",
        width=4,
    )
annotated_path = OUTPUT_DIR / "ocr_boxes.png"
annotated.save(annotated_path)

annotated_preview = annotated.copy()
annotated_preview.thumbnail((650, 800))
print("1) 원본 위 OCR 탐지 영역")
display(annotated_preview)

result_table = pd.DataFrame(
    [
        {
            "OCR 글자": item["text"],
            "신뢰도": round(float(item["confidence"] or 0), 3),
        }
        for item in OCR_RESULT
        if item.get("text")
    ]
)
print("2) 인식한 글자와 신뢰도")
display(result_table)
print(
    "3) 확인할 곳: 기울어진 글자도 선이 따라가는지, "
    "낮은 신뢰도 글자가 틀렸는지"
)

output = {
    "result_source": RESULT_SOURCE,
    "ocr_error": OCR_ERROR,
    "input_file": DISPLAY_INPUT_FILE_NAME,
    "image_size": {
        "width": annotated.width,
        "height": annotated.height,
    },
    "ocr_coordinate_size": {
        "width": OCR_COORDINATE_SIZE[0],
        "height": OCR_COORDINATE_SIZE[1],
    },
    "items": [
        {**item, "matches_source": None, "review_note": ""}
        for item in OCR_RESULT
    ],
}
output_path = OUTPUT_DIR / "ocr_result.json"
output_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
result_label = RESULT_SOURCE
print(
    "✅ 실습 완료:",
    result_label,
    output_path,
    annotated_path,
)

complete_lab_step(4, 7, '`✅ 실습 완료`와 두 결과 파일을 확인합니다.')


## 내가 직접 채우는 3줄

금액·날짜처럼 원본 대조가 필요한 OCR 토큰을 키워드로 표시합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `review_keywords`의 세 `None`만 원본 대조에 사용할 문자열로 바꿉니다. OCR 원문에서 반드시 찾아야 할 값을 고르는
# 연습입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 7, '내 원본 대조 기준 입력', 'OCR 원문에서 반드시 확인할 키워드 세 개를 정합니다.', '빈칸 여부 또는 내가 입력한 키워드가 표시되어야 합니다.', '`review_keywords`의 세 `None`만 원본 대조에 사용할 문자열로 바꿉니다. OCR 원문에서 반드시 찾아야 할 값을 고르는 연습입니다.', 'required')

# TODO: 원본 대조할 키워드 세 개를 넣으세요.
review_keywords = [None, None, None]
if any(value is None for value in review_keywords):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(5, 7, '빈칸 여부 또는 내가 입력한 키워드가 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

날짜의 연도, 합계 라벨, 합계 금액처럼 영향이 큰 토큰을 고릅니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `ANSWER_REVIEW_KEYWORDS`는 공개 정답이고 `zipfile`은 JSON과 위치 이미지를 하나의 다운로드 파일로 묶습니다. 수정
# 없이 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 7, 'OCR 결과 묶음 완성', '공개 정답과 비교하고 OCR 산출물을 ZIP으로 묶습니다.', '대조 결과와 `lesson02_ocr_outputs.zip` 경로를 확인합니다.', '`ANSWER_REVIEW_KEYWORDS`는 공개 정답이고 `zipfile`은 JSON과 위치 이미지를 하나의 다운로드 파일로 묶습니다. 수정 없이 실행합니다.', 'none')

ANSWER_REVIEW_KEYWORDS = ["이태리", "2025", "76,000"]
marked = 0
for item in output["items"]:
    if any(
        keyword in item.get("text", "")
        for keyword in ANSWER_REVIEW_KEYWORDS
    ):
        item["review_note"] = "원본 대조 필수"
        marked += 1
output_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert marked == 3
print("전체 정답 · 원본 대조 표시:", marked, "개")
import zipfile
bundle_path = OUTPUT_DIR / "lesson02_ocr_outputs.zip"
with zipfile.ZipFile(bundle_path, "w") as archive:
    archive.write(output_path, output_path.name)
    archive.write(annotated_path, annotated_path.name)
print("한 번만 다운로드할 묶음:", bundle_path)
download_artifact(bundle_path)

complete_lab_step(6, 7, '대조 결과와 `lesson02_ocr_outputs.zip` 경로를 확인합니다.')


## 선택 실험: 다른 자료로 한 번 더 확인하기

필수 실습을 먼저 끝낸 뒤, 인터넷에서 찾은 공개 문서나 개인정보를
가린 자료 한 장으로 같은 과정을 반복합니다. 결과가 잘 나오지 않아도
실패한 위치와 다음 질문을 남기면 실험이 완료됩니다.


In [ ]:
#@title 🔵 선택: 다른 자료 실험 기록 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# 이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(7, 7, '다른 자료 실험 기록', '인터넷에서 찾은 공개 자료나 비식별 자료의 결과를 네 줄로 정리합니다.', '`lesson02_research_note.md` 파일과 기록 내용이 표시되어야 합니다.', '이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.', 'optional')

# RESEARCH_NOTE_CELL
# TODO(선택): 다른 자료로 다시 실험했다면 아래 입력칸만 채우세요.
자료_구분 = "제공 예제" #@param ["제공 예제", "공개 웹 자료", "비식별 개인 자료", "회사 승인 자료"]
자료_이름_또는_URL = "" #@param {type:"string"}
문서_종류 = "영수증" #@param ["영수증", "견적서", "신청서", "거래명세서", "표 캡처", "기타"]
잘된_점 = "" #@param {type:"string"}
실패한_점 = "" #@param {type:"string"}
다음_질문 = "" #@param {type:"string"}

research_focus = '자료를 바꿨을 때 글자 위치·인식값·신뢰도가 어떻게 달라졌는지 기록합니다.'
note = f'''# {문서_종류} 실험 기록

- 자료 구분: {자료_구분}
- 자료 이름 또는 원문 URL: {자료_이름_또는_URL or "미입력"}
- 이번 교시 관찰 질문: {research_focus}
- 잘된 점: {잘된_점 or "미입력"}
- 실패하거나 이상한 점: {실패한_점 or "미입력"}
- 다음에 바꿔 볼 한 가지: {다음_질문 or "미입력"}
'''
note_path = OUTPUT_DIR / "lesson02_research_note.md"
note_path.write_text(note + "\n", encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(note))
except ImportError:
    print(note)
print("실험 기록 저장:", note_path)

complete_lab_step(7, 7, '`lesson02_research_note.md` 파일과 기록 내용이 표시되어야 합니다.')
